# GDAL S3 Imagery Download with Metadata Preservation

This notebook demonstrates how to download geospatial imagery from AWS S3 using GDAL's Virtual File System (`/vsis3/`) while **preserving original metadata** and optimization settings.

**Features:**
- Direct S3 access using GDAL's Virtual File System  
- **Smart metadata preservation** (compression, tiling, interleaving)
- Support for both public NZ datasets and private S3 buckets
- Multiple authentication methods
- Performance timing and monitoring
- Error handling and status reporting

**Requirements:**
- GDAL >= 3.0 (with S3 support)
- Python >= 3.8

In [ ]:
# Import required libraries
import setup_gdal_env  # Configure GDAL environment
from osgeo import gdal
import os
import time
from datetime import datetime

## Configuration Options

GDAL provides flexible configuration for accessing both public and private S3 buckets. The following sections show different authentication approaches.

In [ ]:
# =============================================================================
# CONFIGURATION: PUBLIC S3 bucket access (NZ datasets)
# =============================================================================

# Configure GDAL for AWS S3 public bucket access
gdal.SetConfigOption('AWS_REGION', 'ap-southeast-2')
gdal.SetConfigOption('AWS_NO_SIGN_REQUEST', 'YES')  # For public buckets
gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN', 'YES')  # Performance optimization

print("✅ Configured for NZ public datasets")
print(f"Region: ap-southeast-2")
print(f"Public access: enabled")

## Private S3 Authentication (Optional)

If you need to access private S3 buckets, uncomment and configure one of the following methods:

### Method A: Direct Credentials
```python
gdal.SetConfigOption('AWS_ACCESS_KEY_ID', 'your_access_key_here')
gdal.SetConfigOption('AWS_SECRET_ACCESS_KEY', 'your_secret_key_here') 
gdal.SetConfigOption('AWS_REGION', 'your-region')
```

### Method B: AWS Profile (Recommended)
```python
gdal.SetConfigOption('AWS_PROFILE', 'your_profile_name')
gdal.SetConfigOption('AWS_REGION', 'your-region')
```

### Method C: Environment Variables
Set these before running:
```bash
export AWS_ACCESS_KEY_ID=your_access_key
export AWS_SECRET_ACCESS_KEY=your_secret_key  
export AWS_DEFAULT_REGION=your_region
```

In [ ]:
# =============================================================================
# PRIVATE S3 CONFIGURATION (Uncomment if needed)
# =============================================================================

# Example configurations for private buckets:
"""
# Method A: Using AWS credentials directly
gdal.SetConfigOption('AWS_ACCESS_KEY_ID', 'your_access_key_here')
gdal.SetConfigOption('AWS_SECRET_ACCESS_KEY', 'your_secret_key_here') 
gdal.SetConfigOption('AWS_REGION', 'your-region')  # e.g., 'us-east-1'

# Method B: Using AWS profile (reads from ~/.aws/credentials)
gdal.SetConfigOption('AWS_PROFILE', 'your_profile_name')
gdal.SetConfigOption('AWS_REGION', 'your-region')

# Additional private S3 options:
gdal.SetConfigOption('AWS_REQUEST_PAYER', 'requester')  # If bucket requires requester pays
gdal.SetConfigOption('AWS_S3_ENDPOINT', 'custom-endpoint.com')  # For non-AWS S3-compatible services
"""

print("⚠️  Private S3 configuration is commented out")
print("📝 Uncomment and edit the above section if you need private bucket access")

## Setup Output Directory

Ensure the output directory exists before downloading.

In [ ]:
# Setup output directory
output_dir = "c:\\data\\imagery"
os.makedirs(output_dir, exist_ok=True)

print(f"📁 Output directory: {output_dir}")
print(f"✅ Directory exists: {os.path.exists(output_dir)}")

## Download Imagery from S3 with Metadata Preservation

This section demonstrates downloading a sample image from the NZ public imagery dataset using `gdal.Translate()` while **preserving original metadata** such as compression, tiling, and interleaving.

**NZ Dataset Information:**
- **Bucket**: `nz-imagery` 
- **Region**: `ap-southeast-2`
- **Content**: High-resolution aerial imagery (RGB, multispectral)
- **Example**: Taranaki region 10cm resolution RGB imagery

**Metadata Preservation Features:**
- 📦 **Compression** - LZW, JPEG, etc.
- 🔳 **Tiling** - Maintains COG optimization  
- 🎨 **Interleaving** - Pixel vs band organization

In [ ]:
# Start timing
print(f"🚀 Starting download at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
start_time = time.time()

# Define source and destination
source_url = "/vsis3/nz-imagery/taranaki/taranaki_2022-2023_0.1m/rgb/2193/BH28_500_095032.tiff"
destination_file = "c:\\data\\imagery\\taranaki_sample.tiff"

print(f"📥 Source: s3://nz-imagery/taranaki/...")
print(f"📦 Destination: {destination_file}")
print(f"⏳ Downloading...")

## Metadata Reading and Preservation

This is the key improvement - we first read the source metadata to understand its optimization settings, then preserve them in the output file.

In [ ]:
# =============================================================================
# METADATA PRESERVATION: Read source properties and preserve them
# =============================================================================

print("🔍 Reading source metadata...")
src = gdal.Open(source_url)
if src is None:
    print("❌ Failed to open source file")
    raise RuntimeError("Could not open source dataset")

# Read IMAGE_STRUCTURE metadata to preserve compression and tiling
img_md = src.GetMetadata("IMAGE_STRUCTURE")
creation_opts = []

# Preserve compression if present
if "COMPRESSION" in img_md:
    compression = img_md["COMPRESSION"]
    creation_opts.append(f"COMPRESS={compression}")
    print(f"📦 Preserving compression: {compression}")

# Preserve tiling if present
if img_md.get("TILED") == "YES":
    creation_opts.append("TILED=YES")
    print("🔳 Preserving tiled structure")

# Preserve interleaving if pixel interleaved
if img_md.get("INTERLEAVE") == "PIXEL":
    creation_opts.append("INTERLEAVE=PIXEL")
    print("🎨 Preserving pixel interleaving")

# Show what creation options will be used
if creation_opts:
    print(f"⚙️  Using creation options: {creation_opts}")
else:
    print("📋 No special creation options needed")

print(f"📥 Downloading with metadata preservation...")

In [ ]:
# Download S3 file to local file with preserved metadata
# Key change: using opened dataset (src) instead of URL string
# and passing creationOptions to preserve metadata
result = gdal.Translate(
    destination_file,     # destination (local file)
    src,                  # source (opened dataset)
    creationOptions=creation_opts  # preserve metadata
)

# Calculate timing
end_time = time.time()
duration = end_time - start_time

print(f"⏱️  Finished download at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"⚡ Download duration: {duration:.2f} seconds")

In [ ]:
# Verify metadata preservation in downloaded file
try:
    local_ds = gdal.Open(destination_file)
    if local_ds:
        print("✅ Verifying downloaded file metadata preservation:")
        print(f"   Destination file: {destination_file}")
        print(f"   Size: {os.path.getsize(destination_file):,} bytes\n")
        
        # Check if key metadata was preserved
        local_metadata = local_ds.GetMetadata("IMAGE_STRUCTURE")
        
        if local_metadata:
            print("📁 Preserved metadata:")
            for key, value in local_metadata.items():
                print(f"   {key}: {value}")
        else:
            print("⚠️  No IMAGE_STRUCTURE metadata found in local file")
        
        print(f"\n🎯 File format: {local_ds.GetDriver().ShortName}")
        local_ds = None
    else:
        print("❌ Failed to open downloaded file")
        
except Exception as e:
    print(f"❌ Error verifying downloaded file: {str(e)}

## Advanced Examples

### Custom Creation Options

You can customize the creation options for advanced use cases:

In [ ]:
# Example: Using custom creation options for specific needs
# This demonstrates how to customize the metadata preservation

# High compression option
custom_creation_opts = [
    'COMPRESS=JPEG',
    'QUALITY=85',
    'TILED=YES',
    'BLOCKXSIZE=512',
    'BLOCKYSIZE=512',
    'PHOTOMETRIC=YCBCR'
]

print("🔧 Custom creation options example:")
print(f"   Options: {custom_creation_opts}")
print("   Note: These options create a highly compressed JPEG-in-TIFF format")
print("         Suitable for high resolution imagery where file size is critical")

# Example: Format conversion during download
print("\n📝 Example workflow for format conversion:")
print("   1. Read metadata from S3 TIFF")
print("   2. Apply format-specific optimizations")
print("   3. Download with custom compression")
print("   4. Verify metadata preservation")

## Summary

This notebook demonstrates how to:

1. **Read raster metadata** from S3-hosted TIFF files using GDAL's Virtual File System
2. **Preserve metadata** during download by using creation options
3. **Verify preservation** by comparing source and destination metadata
4. **Optimize for Cloud-Optimized GeoTIFF (COG)** compliance

### Key Benefits:
- Direct S3 access without local downloading for metadata
- Intelligent metadata preservation
- COG-compliant downloads
- Efficient for large imagery datasets

### Next Steps:
- Explore the companion script [imagery_gdal_read.py](../imagery_gdal_read.py)
- Review GDAL documentation in [README_GDAL.md](../README_GDAL.md)
- Try the analysis tool [aws_gdal_raster_info.py](../aws_gdal_raster_info.py)

In [ ]:
# Check results and provide feedback
if result:
    # Get file size for additional info
    file_size = os.path.getsize(destination_file) if os.path.exists(destination_file) else 0
    file_size_mb = file_size / (1024 * 1024)
    
    print("✅ File downloaded successfully!")
    print(f"📄 File: {destination_file}")
    print(f"📊 Size: {file_size_mb:.2f} MB")
    print(f"🚀 Speed: {file_size_mb/duration:.2f} MB/s")
    
    # Get basic image info using GDAL
    dataset = gdal.Open(destination_file)
    if dataset:
        print(f"🖼️  Dimensions: {dataset.RasterXSize} x {dataset.RasterYSize}")
        print(f"📡 Bands: {dataset.RasterCount}")
        print(f"🗂️  Format: {dataset.GetDriver().LongName}")
        dataset = None  # Close dataset
else:
    print("❌ Failed to download file")
    print("🔍 Check your configuration and network connection")

## Available NZ Datasets

The following public datasets are available from NZ (Land Information New Zealand):

| Dataset | Bucket | Region | Content |
|---------|--------|---------|---------|
| **Imagery** | `nz-imagery` | `ap-southeast-2` | Aerial imagery, RGB, multispectral |
| **Elevation** | `nz-elevation` | `ap-southeast-2` | Digital elevation models (DEMs) |
| **Coastal** | `nz-coastal` | `ap-southeast-2` | Coastal datasets and bathymetry |

### Example File Paths:
```
# High-resolution RGB imagery (10cm resolution)
/vsis3/nz-imagery/taranaki/taranaki_2022-2023_0.1m/rgb/2193/BH28_500_095032.tiff

# Elevation data
/vsis3/nz-elevation/wellington/wellington_2019_1m/dem_1m/2193/BQ31_10000_0101.tif

# Path structure: region/survey/resolution/type/grid-reference.tif
```

In [ ]:
# Example: Download from different NZ dataset (elevation)
# Uncomment and run to download an elevation model

"""
elevation_source = "/vsis3/nz-elevation/canterbury/canterbury_2018-2019_1m/dem_1m/2193/BX24_10000_0101.tif"
elevation_dest = "c:\\data\\imagery\\canterbury_dem.tif"

print("🏔️  Downloading elevation data...")
start_time = time.time()

result = gdal.Translate(elevation_dest, elevation_source)

if result:
    duration = time.time() - start_time
    file_size_mb = os.path.getsize(elevation_dest) / (1024 * 1024)
    print(f"✅ Elevation model downloaded successfully!")
    print(f"📄 File: {elevation_dest}")
    print(f"📊 Size: {file_size_mb:.2f} MB")
    print(f"⏱️  Duration: {duration:.2f} seconds")
"""

print("💡 Uncomment the above code to download elevation data")

## Advanced Usage Examples

### Format Conversion
GDAL can convert formats during download:

In [ ]:
# Example: Convert TIFF to PNG during download
# Uncomment to run:

"""
png_dest = "c:\\data\\imagery\\sample_converted.png"

result = gdal.Translate(
    png_dest,
    source_url,
    format="PNG"  # Convert to PNG format
)

if result:
    print(f"✅ Converted to PNG: {png_dest}")
"""

print("💡 Uncomment above to convert TIFF to PNG during download")

In [ ]:
# Example: Add compression and tiling options
# Uncomment to run:

"""
compressed_dest = "c:\\data\\imagery\\sample_compressed.tiff"

result = gdal.Translate(
    compressed_dest,
    source_url,
    creationOptions=["COMPRESS=LZW", "TILED=YES"]  # Add compression and tiling
)

if result:
    print(f"✅ Compressed TIFF created: {compressed_dest}")
"""

print("💡 Uncomment above to create compressed TIFF with tiling")

## Troubleshooting

### Common Issues

**"NULL pointer" Error**
- Check parameter order: `gdal.Translate(destination, source)`
- Verify AWS configuration and bucket permissions

**Slow Downloads** 
- Already optimized with `GDAL_DISABLE_READDIR_ON_OPEN='YES'`
- Check network connection and S3 region

**Permission Issues**
- For public buckets: Ensure `AWS_NO_SIGN_REQUEST='YES'` is set
- For private buckets: Check credentials and bucket permissions

### Debug Mode
To enable debug output:

In [ ]:
# Enable GDAL debug mode (uncomment if needed)
# gdal.SetConfigOption('CPL_DEBUG', 'ON')
# gdal.SetConfigOption('CPL_CURL_VERBOSE', 'YES')

print("🔍 Debug mode is disabled by default")
print("💡 Uncomment above lines to enable detailed GDAL logging")

## Summary

This notebook demonstrated:

1. ✅ **GDAL S3 Configuration** - Both public and private bucket access
2. ✅ **Direct Downloads** - Using `gdal.Translate()` with S3 URLs  
3. ✅ **Performance Monitoring** - Timing and file size reporting
4. ✅ **NZ Datasets** - Access to New Zealand geospatial data
5. ✅ **Format Conversion** - Converting between formats during download
6. ✅ **Error Handling** - Robust download verification

**Next Steps:**
- Try downloading different regions or datasets
- Experiment with format conversions
- Set up private bucket access if needed
- Explore other GDAL utilities for geospatial processing

**Related Notebooks:**
- `01_imagery_aws_read.ipynb` - obstore-based approach
- `02_imagery_rasterio_read.ipynb` - rasterio-based approach  
- `05_gdal_utilities.ipynb` - Additional GDAL utilities